# Object Detection with a Pretrained Faster R-CNN

This notebook detects objects in an image using a Faster R-CNN model pretrained on the COCO dataset (from torchvision). It draws bounding boxes and labels for every detection above a confidence threshold. The original attempt used OpenCV with a local model file and `cv2.imshow`, which does not work in Colab; this version uses a ready-made pretrained model instead.

**How to run:** Open in Google Colab (GPU optional; CPU works for one image) and run all cells. Pretrained weights download automatically.

_Rewritten from my original Colab notebook: `Untitled0.ipynb`._

## 1. Imports

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from skimage import data
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights
from torchvision.transforms.functional import to_tensor

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 2. Load the pretrained model

In [ ]:
weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
model = fasterrcnn_mobilenet_v3_large_fpn(weights=weights).to(device).eval()
class_labels = weights.meta["categories"]  # COCO class names

## 3. Load an image

We use a sample photo bundled with scikit-image. To use your own image, replace it with `np.array(PIL.Image.open('my_image.jpg').convert('RGB'))`.

In [ ]:
image = data.astronaut()  # RGB numpy array, shape (H, W, 3)
plt.imshow(image); plt.axis("off"); plt.show()

## 4. Run detection

In [ ]:
with torch.no_grad():
    output = model([to_tensor(image).to(device)])[0]

THRESHOLD = 0.5
keep = output["scores"] > THRESHOLD
boxes = output["boxes"][keep].cpu().numpy()
labels = output["labels"][keep].cpu().numpy()
scores = output["scores"][keep].cpu().numpy()
print(f"Found {len(boxes)} objects above {THRESHOLD}")

## 5. Draw the results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image)
for (x_min, y_min, x_max, y_max), label, score in zip(boxes, labels, scores):
    ax.add_patch(patches.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
                                   fill=False, color="lime", linewidth=2))
    ax.text(x_min, y_min - 5, f"{class_labels[label]} {score:.2f}", color="black",
            backgroundcolor="lime", fontsize=9)
ax.axis("off")
plt.show()